# 01 — GDELT GKG Exploration

Goals:
1. Verify BigQuery connectivity and credentials
2. Inspect the GKG schema (column names and types)
3. Run `explore_gkg.sql` blocks to validate that GenAI entity names appear in `AllNames` and `Quotations`
4. Identify which `V2Themes` codes co-occur with AI entity mentions (to refine governance filter)
5. Estimate corpus size before running the full extraction

In [ ]:
import os
from pathlib import Path

import pandas as pd
from google.cloud import bigquery

# Set credentials if not already set in environment
# os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = str(Path('../') / 'service-account.json')
# os.environ['BIGQUERY_PROJECT'] = 'your-project-id'

PROJECT = os.environ.get('BIGQUERY_PROJECT', 'your-project-id')
client = bigquery.Client(project=PROJECT)
print('Connected to BigQuery project:', PROJECT)

## 1. Schema inspection

In [ ]:
table = client.get_table('gdelt-bq.gdeltv2.gkg_partitioned')
schema_df = pd.DataFrame([
    {'name': f.name, 'field_type': f.field_type, 'mode': f.mode}
    for f in table.schema
])
schema_df

## 2. Sample records mentioning 'chatgpt'

In [ ]:
sample_sql = open('../queries/explore_gkg.sql').read().split('-- ====')[1].split('-- ====')[0]
# Or paste Query 1 directly:
sample_sql = """
SELECT
  DATE,
  SourceCommonName,
  DocumentIdentifier,
  LEFT(AllNames, 300)     AS AllNames_preview,
  LEFT(Quotations, 400)   AS Quotations_preview,
  LEFT(V2Themes, 300)     AS V2Themes_preview,
  V2Tone,
  LEFT(V2Locations, 200)  AS V2Locations_preview
FROM `gdelt-bq.gdeltv2.gkg_partitioned`
WHERE DATE(_PARTITIONTIME) >= '2023-01-01'
  AND DATE(_PARTITIONTIME) <  '2023-02-01'
  AND LOWER(AllNames) LIKE '%chatgpt%'
LIMIT 20
"""
sample_df = client.query(sample_sql).to_dataframe()
sample_df

## 3. Top V2Themes co-occurring with AI entity mentions

In [ ]:
themes_sql = """
SELECT
  theme,
  COUNT(*) AS n
FROM `gdelt-bq.gdeltv2.gkg_partitioned`,
  UNNEST(SPLIT(V2Themes, ';')) AS theme
WHERE DATE(_PARTITIONTIME) >= '2023-01-01'
  AND DATE(_PARTITIONTIME) <  '2023-07-01'
  AND LOWER(AllNames) LIKE '%chatgpt%'
  AND theme != ''
GROUP BY theme
ORDER BY n DESC
LIMIT 100
"""
themes_df = client.query(themes_sql).to_dataframe()
themes_df.head(30)

## 4. Monthly volume quick check

In [ ]:
import matplotlib.pyplot as plt

vol_sql = """
SELECT
  DATE(DATE_TRUNC(PARSE_TIMESTAMP('%Y%m%d%H%M%S', CAST(DATE AS STRING)), MONTH)) AS month,
  COUNT(*) AS n_articles
FROM `gdelt-bq.gdeltv2.gkg_partitioned`
WHERE DATE(_PARTITIONTIME) >= '2022-11-01'
  AND DATE(_PARTITIONTIME) <  '2026-07-01'
  AND (
    LOWER(AllNames) LIKE '%chatgpt%'
    OR LOWER(AllNames) LIKE '%openai%'
    OR LOWER(AllNames) LIKE '%gemini%'
    OR LOWER(AllNames) LIKE '%large language model%'
    OR LOWER(AllNames) LIKE '%gpt-4%'
  )
GROUP BY month
ORDER BY month
"""
vol_df = client.query(vol_sql).to_dataframe()
vol_df['month'] = pd.to_datetime(vol_df['month'])

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(vol_df['month'], vol_df['n_articles'])
ax.set_title('Monthly AI entity mentions in GDELT GKG (GenAI only, no governance filter)')
ax.set_xlabel('Month')
ax.set_ylabel('Article count')
plt.tight_layout()
plt.show()

## 5. Notes and decisions

After running the above:
- Record which `V2Themes` codes appear most frequently alongside AI mentions
- Add any relevant new theme codes to the governance filter in `extract_genai_gov.sql`
- Verify that `Quotations` actually contains governance language in sample records
- Estimate full corpus size before running the expensive extraction